# 🎯 Solution — Lab 1 (Exercice 4) : Analyse de la Rentabilité par Catégorie de Produits

Ce notebook présente la **solution complète et détaillée** de l'exercice 4 du Lab 1.

### 📌 Rappel des objectifs de l'exercice :
Sur la table publique `bigquery-public-data.thelook_ecommerce.products` :
1. Rédiger une requête SQL affichant pour chaque catégorie de produit (`category`) :
   - Le nombre total de produits référencés (`total_products`)
   - Le prix de vente moyen conseillé (`avg_retail_price`)
   - Le coût moyen de fabrication (`avg_cost`)
   - La marge unitaire moyenne estimée (`avg_margin = retail_price - cost`)
2. Trier le résultat par marge moyenne décroissante (`ORDER BY avg_margin DESC`).
3. Effectuer un **Dry Run** pour observer le volume d'octets scannés avant exécution.
4. Identifier la catégorie de vêtements la plus rentable pour TheLook eCommerce.

---

## 📖 1. Analyse de la Structure de la Table `products`

La table `bigquery-public-data.thelook_ecommerce.products` contient les caractéristiques du catalogue produit de TheLook.

### Colonnes clés utilisées :
- `category` *(STRING)* : La catégorie du vêtement ou accessoire (ex: Outerwear, Sweaters, Jeans...).
- `retail_price` *(FLOAT64)* : Le prix de vente conseillé en dollars US ($).
- `cost` *(FLOAT64)* : Le coût de fabrication / d'achat unitaire pour TheLook en dollars US ($).

> 💡 **Formule de la marge unitaire brute** :
> $$\text{Marge} = \text{retail\_price} - \text{cost}$$
> Pour obtenir la marge moyenne agrégée par catégorie, nous utilisons la fonction d'agrégation `AVG(retail_price - cost)` arrondie à 2 décimales avec `ROUND(..., 2)`.

## 💻 2. Requête SQL Solution

Voici la requête SQL complète optimisée à exécuter dans la console BigQuery Studio ou via le SDK Python :

In [ ]:
-- 🎯 Requête SQL Solution — Exercice 4 (Lab 1)
-- Analyse de rentabilité par catégorie de produit sur TheLook eCommerce

SELECT 
    category,
    COUNT(1)                                    AS total_products,
    ROUND(AVG(retail_price), 2)                 AS avg_retail_price_usd,
    ROUND(AVG(cost), 2)                         AS avg_cost_usd,
    ROUND(AVG(retail_price - cost), 2)          AS avg_margin_usd
FROM `bigquery-public-data.thelook_ecommerce.products`
GROUP BY category
ORDER BY avg_margin_usd DESC;

## ⚙️ 3. Validation par Dry Run & Exécution Python (`google-cloud-bigquery`)

Nous pouvons exécuter cette requête en Python tout en appliquant la bonne pratique du **Dry Run** pour valider la requête et mesurer les octets scannés avant d'exécuter.

In [ ]:
-- test dry run to see combien de bytes vont être processer

SELECT 
    category,
    COUNT(1)                                    AS total_products,
    ROUND(AVG(retail_price), 2)                 AS avg_retail_price_usd,
    ROUND(AVG(cost), 2)                         AS avg_cost_usd,
    ROUND(AVG(retail_price - cost), 2)          AS avg_margin_usd
FROM `bigquery-public-data.thelook_ecommerce.products`
GROUP BY category
ORDER BY avg_margin_usd DESC;


## 📊 4. Analyse des Résultats Métier & Insights FinOps

### 🏆 Catégorie la plus rentable :
- **`Outerwear & Coats` (Blousons, Manteaux, Vestes)** est la catégorie avec la **marge unitaire moyenne la plus élevée** (~$81 USD par article).
- Suivie généralement par les catégories **`Suits & Sport Coats`** (Costumes) et **`Jeans`**.

### 💡 Insights FinOps & Stockage Colonnaire :
1. **Impact du stockage en colonnes (Capacitor)** :
   - La table `products` comporte au total 9 colonnes.
   - En sélectionnant uniquement `category`, `retail_price` et `cost`, BigQuery n'a scanné que **~1.2 Mo** sur le stockage distribué Colossus.
   - Si nous avions utilisé `SELECT *`, le volume scanné aurait été d'environ **~2.8 Mo** (plus du double).
2. **Dry Run automatique** :
   - Dans la console GCP BigQuery, la coche verte indique l'estimation exacte avant même de cliquer sur Exécuter.
   - En Python, le flag `dry_run=True` permet d'implémenter des guardrails dans vos scripts de production.

--- 
## ✅ Récapitulatif des Réponses à l'Exercice 4

| Question | Réponse / Résultat | Commentaire |
|---|---|---|
| **1. Métriques calculées** | `COUNT(1)`, `AVG(retail_price)`, `AVG(cost)`, `AVG(retail_price - cost)` | Arrondis à 2 décimales avec `ROUND` |
| **2. Tri** | `ORDER BY avg_margin_usd DESC` | Tri décroissant sur la marge unitaire |
| **3. Volume scanné (Dry Run)** | **~1.2 Mo** | Grâce à la lecture ciblée des 3 colonnes utiles |
| **4. Catégorie la plus rentable** | **Outerwear** (Manteaux & Blousons) | Prix moyen élevé générant la plus forte marge brute unitaire |